# Smoke — checkpoint YOLOX của DanceTrack (nguồn detections b)

Mục đích: kiểm **trước khi dùng** xem `noahcao/dancetrack_models/bytetrack_models/bytetrack_model.pth.tar` có (1) tải được và nạp `strict=True` vào YOLOX-X, (2) ra box người trên một khung val, (3) tự khai dữ liệu train hay không. Không chấm điểm, không chạy tracker.

Tiêu chí đạt/không đạt nằm ở `detection_source_b.smoke_acceptance` trong `tracker-search-space-dancetrack-261027.json`. Notebook chỉ báo dữ kiện; **việc coi val là "chưa thấy" hay bỏ nguồn (b) do người quyết định** sau khi đọc `yolox-checkpoint-smoke.json`.

Chạy tuần tự. GPU không bắt buộc (một khung, CPU vài giây). Cần ~1 GB trống cho checkpoint.

In [1]:
import subprocess, sys, json
from pathlib import Path

IN_COLAB = Path('/content').exists()
WORK = Path('/content/yolox-smoke') if IN_COLAB else Path.cwd() / 'yolox-smoke-work'
OUT = Path('/content/drive/MyDrive/DetTrack/yolox-smoke-261027') if IN_COLAB else WORK / 'out'
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
for p in (WORK, OUT):
    p.mkdir(parents=True, exist_ok=True)
# yolox is NOT pip-installed (its setup.py imports torch at build time and failed on Colab py3.13); inference only needs the
# python sources of yolox.models / yolox.utils, so clone the tagged release and put it on sys.path.
subprocess.run([sys.executable, '-m', 'pip', 'install', 'huggingface_hub', 'fsspec', 'aiohttp', 'opencv-python-headless',
                'loguru', 'thop', 'tabulate', 'pycocotools'], check=True)
YOLOX_DIR = WORK / 'YOLOX'
if not YOLOX_DIR.exists():
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', '0.3.0', 'https://github.com/Megvii-BaseDetection/YOLOX.git',
                    str(YOLOX_DIR)], check=True)
sys.path.insert(0, str(YOLOX_DIR))
import yolox, yolox.models, yolox.utils      # fail here, not inside the smoke subprocess, if a dependency is missing
print('yolox', yolox.__version__, 'from', Path(yolox.__file__).parent, '\nwork:', WORK, '\nresults:', OUT)

Mounted at /content/drive
yolox 0.3.0 from /content/yolox-smoke/YOLOX/yolox 
work: /content/yolox-smoke 
results: /content/drive/MyDrive/DetTrack/yolox-smoke-261027


In [2]:
SCRIPT = WORK / 'yolox_checkpoint_smoke.py'
SCRIPT.write_text('"""Colab smoke for detection source (b): can the DanceTrack YOLOX checkpoint be loaded and does it detect the dancers?\n\nRun in Colab (GPU optional). Checks only what `detection_source_b.smoke_acceptance` lists: load + sha256, boxes that\nagree with the GT of a few val frames, and what the checkpoint says about its own training. It never claims val is\nunseen; the training-data question stays open unless the checkpoint or the upstream exp file settles it.\n\nTwo preprocessings are run because the right one is not assumed: the one that agrees with GT is reported.\n\n    PYTHONPATH=<YOLOX 0.3.0 source> python yolox_checkpoint_smoke.py --frames f1.jpg f2.jpg --gt gt.json\n"""\nimport argparse\nimport hashlib\nimport json\nimport sys\nimport time\nfrom pathlib import Path\n\nREPO, FILE = "noahcao/dancetrack_models", "bytetrack_models/bytetrack_model.pth.tar"\nEXPECTED_BYTES = 792_835_731\nVARIANTS = ("raw_bgr", "imagenet_rgb")\n\n\ndef sha256(path, chunk=1 << 24):\n    h = hashlib.sha256()\n    with open(path, "rb") as f:\n        while block := f.read(chunk):\n            h.update(block)\n    return h.hexdigest()\n\n\ndef describe_checkpoint(ckpt):\n    """Metadata a checkpoint carries about itself; absent keys are reported as absent, not guessed."""\n    state = ckpt.get("model", ckpt)\n    head = {k: tuple(v.shape) for k, v in state.items() if "cls_preds" in k}\n    stem = next((tuple(v.shape) for k, v in state.items() if k.endswith("stem.conv.conv.weight")), None)\n    return dict(\n        top_level_keys=sorted(ckpt.keys()) if isinstance(ckpt, dict) else None,\n        n_tensors=len(state), stem_conv_shape=stem, cls_pred_shapes=head,\n        start_epoch=ckpt.get("start_epoch"),\n        # yolox_x has stem out-channels 80 (width 1.25 * 64); num_classes = cls_preds out-channels\n        num_classes=next(iter(head.values()))[0] if head else None,\n        has_exp_or_train_args=[k for k in ("exp", "args", "exp_name", "dataset") if isinstance(ckpt, dict) and k in ckpt])\n\n\ndef iou_matrix(a, b):\n    """a: Nx4, b: Mx4 as x1,y1,x2,y2."""\n    import numpy as np\n    a, b = np.asarray(a, float).reshape(-1, 4), np.asarray(b, float).reshape(-1, 4)\n    lt, rb = np.maximum(a[:, None, :2], b[None, :, :2]), np.minimum(a[:, None, 2:], b[None, :, 2:])\n    inter = np.prod(np.clip(rb - lt, 0, None), axis=2)\n    area = lambda x: (x[:, 2] - x[:, 0]) * (x[:, 3] - x[:, 1])\n    return inter / (area(a)[:, None] + area(b)[None] - inter + 1e-9)\n\n\ndef build_model(state):\n    import torch\n    from yolox.models import YOLOX, YOLOPAFPN, YOLOXHead\n    depth, width, ncls = 1.33, 1.25, 1  # yolox_x; checked against the checkpoint shapes by load_state_dict(strict=True)\n    model = YOLOX(YOLOPAFPN(depth, width, in_channels=[256, 512, 1024]), YOLOXHead(ncls, width, in_channels=[256, 512, 1024]))\n    model.load_state_dict(state, strict=True)\n    return model.to("cuda" if torch.cuda.is_available() else "cpu").eval(), ncls\n\n\ndef preprocess(img, size, variant):\n    """raw_bgr: letterbox 114, BGR 0-255 (original YOLOX eval). imagenet_rgb: RGB /255 then ImageNet mean/std."""\n    import cv2\n    import numpy as np\n    scale = min(size[0] / img.shape[0], size[1] / img.shape[1])\n    resized = cv2.resize(img, (int(img.shape[1] * scale), int(img.shape[0] * scale)))\n    canvas = 114 * np.ones((*size, 3), dtype="uint8")\n    canvas[:resized.shape[0], :resized.shape[1]] = resized\n    x = canvas.astype("float32")\n    if variant == "imagenet_rgb":\n        x = x[:, :, ::-1] / 255.0\n        x = (x - np.array([0.485, 0.456, 0.406], "float32")) / np.array([0.229, 0.224, 0.225], "float32")\n    return np.ascontiguousarray(x.transpose(2, 0, 1), dtype="float32")[None], scale\n\n\ndef detect(model, ncls, frame_path, gt_xywh, variant, size=(800, 1440), nms=0.7):\n    """Detections at several score thresholds, with precision/recall at IoU 0.5 against the frame GT."""\n    import cv2\n    import numpy as np\n    import torch\n    from yolox.utils import postprocess\n    img = cv2.imread(str(frame_path))\n    assert img is not None, f"cannot read {frame_path}"\n    x, scale = preprocess(img, size, variant)\n    dev = next(model.parameters()).device\n    t = time.time()\n    with torch.no_grad():\n        out = postprocess(model(torch.from_numpy(x).to(dev)), ncls, 0.1, nms)[0]\n    seconds = round(time.time() - t, 3)\n    det = out.cpu().numpy() if out is not None else np.zeros((0, 7))\n    score, boxes = det[:, 4] * det[:, 5], det[:, :4] / scale\n    gt = [[x0, y0, x0 + w, y0 + h] for x0, y0, w, h in gt_xywh]\n    res = dict(seconds=seconds, n_gt=len(gt), by_threshold={})\n    for thr in (0.3, 0.5, 0.7):\n        keep = boxes[score >= thr]\n        if len(keep) and gt:\n            m = iou_matrix(keep, gt)\n            tp_det, tp_gt = int((m.max(1) >= 0.5).sum()), int((m.max(0) >= 0.5).sum())\n        else:\n            tp_det = tp_gt = 0\n        res["by_threshold"][str(thr)] = dict(n_det=int(len(keep)), precision=round(tp_det / max(len(keep), 1), 3),\n                                             recall=round(tp_gt / max(len(gt), 1), 3))\n    return res\n\n\ndef mean_at_half(report, variant, key):\n    rows = [r["by_threshold"]["0.5"][key] for r in report.get("inference", {}).get(variant, {}).values()]\n    return round(sum(rows) / len(rows), 3) if rows else None\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--frames", required=True, nargs="+", help="DanceTrack val frames (jpg)")\n    ap.add_argument("--gt", required=True, help="json {frame file name: [[x,y,w,h], ...]} for those frames")\n    ap.add_argument("--out", default="yolox-checkpoint-smoke.json")\n    a = ap.parse_args()\n    from huggingface_hub import hf_hub_download\n    import torch\n    path = hf_hub_download(REPO, FILE)\n    size = Path(path).stat().st_size\n    report = dict(repo=REPO, file=FILE, bytes=size, bytes_match_registry=size == EXPECTED_BYTES, sha256=sha256(path),\n                  loaded_strict=False)\n    ckpt = torch.load(path, map_location="cpu", weights_only=False)\n    report["checkpoint"] = describe_checkpoint(ckpt)\n    try:\n        model, ncls = build_model(ckpt.get("model", ckpt))\n        report["loaded_strict"] = True\n        gt = json.loads(Path(a.gt).read_text(encoding="utf-8"))\n        report["inference"] = {v: {Path(f).name: detect(model, ncls, f, gt[Path(f).name], v) for f in a.frames}\n                               for v in VARIANTS}\n    except Exception as e:  # report the failure; the verdict is the plan owner\'s call\n        report["error"] = f"{type(e).__name__}: {e}"[:600]\n    agree = {v: dict(precision=mean_at_half(report, v, "precision"), recall=mean_at_half(report, v, "recall"))\n             for v in VARIANTS}\n    report["agreement_with_gt_at_score_0.5"] = agree\n    best = max(agree, key=lambda v: ((agree[v]["recall"] or 0), (agree[v]["precision"] or 0))) if report.get("inference") else None\n    report["verdict_inputs"] = dict(loads=report["loaded_strict"], best_variant=best,\n                                    training_data_documented=bool(report["checkpoint"]["has_exp_or_train_args"]))\n    Path(a.out).write_text(json.dumps(report, indent=2, ensure_ascii=False), encoding="utf-8")\n    print(json.dumps(report, indent=2, ensure_ascii=False))\n    return 0 if report["loaded_strict"] else 1\n\n\nif __name__ == "__main__":\n    sys.exit(main())\n', encoding='utf-8')
print('script written:', SCRIPT)

script written: /content/yolox-smoke/yolox_checkpoint_smoke.py


In [3]:
# One DanceTrack val frame by HTTP range read (does not download the 11 GB archive).
import zipfile, fsspec
VAL_URL = 'https://huggingface.co/datasets/noahcao/dancetrack/resolve/main/val.zip'
N_SEQ = 3                                   # distinct sequences; 3 frames each, evenly spread, deterministic
FRAMES, GT = [], {}
with fsspec.open(VAL_URL, 'rb', block_size=1 << 20) as fh, zipfile.ZipFile(fh) as zf:
    names = [i.filename for i in zf.infolist() if not i.is_dir()]
    seqs = sorted({n.split('/')[1] for n in names if '/img1/' in n})
    for seq in seqs[:: max(1, len(seqs) // N_SEQ)][:N_SEQ]:
        rows = [l.split(',') for l in zf.read(f'val/{seq}/gt/gt.txt').decode().splitlines()]
        n_frames = max(int(r[0]) for r in rows)
        for f in (n_frames // 4, n_frames // 2, 3 * n_frames // 4):
            path = WORK / f'{seq}_{f:08d}.jpg'
            path.write_bytes(zf.read(f'val/{seq}/img1/{f:08d}.jpg'))
            FRAMES.append(str(path))
            GT[path.name] = [[float(r[2]), float(r[3]), float(r[4]), float(r[5])] for r in rows if int(r[0]) == f]
GT_FILE = WORK / 'gt.json'
GT_FILE.write_text(json.dumps(GT))
print(len(FRAMES), 'frames; GT boxes per frame:', {k: len(v) for k, v in GT.items()})

9 frames; GT boxes per frame: {'dancetrack0004_00000300.jpg': 3, 'dancetrack0004_00000601.jpg': 2, 'dancetrack0004_00000902.jpg': 4, 'dancetrack0026_00000075.jpg': 14, 'dancetrack0026_00000151.jpg': 17, 'dancetrack0026_00000226.jpg': 13, 'dancetrack0063_00000250.jpg': 8, 'dancetrack0063_00000500.jpg': 8, 'dancetrack0063_00000750.jpg': 8}


In [4]:
import os
proc = subprocess.run([sys.executable, str(SCRIPT), '--frames', *FRAMES, '--gt', str(GT_FILE),
                       '--out', str(OUT / 'yolox-checkpoint-smoke.json')],
                      capture_output=True, text=True, env={**os.environ, 'PYTHONPATH': str(YOLOX_DIR)})
if proc.returncode:
    print('--- stderr ---\n', proc.stderr[-4000:])
report = json.loads((OUT / 'yolox-checkpoint-smoke.json').read_text(encoding='utf-8'))
print('agreement with GT (IoU 0.5, score >= 0.5, mean over frames):', json.dumps(report['agreement_with_gt_at_score_0.5'], indent=1))
for variant, frames in report.get('inference', {}).items():
    print(variant)
    for name, r in frames.items():
        print(f"  {name}: GT={r['n_gt']}", {t: (v['n_det'], v['precision'], v['recall']) for t, v in r['by_threshold'].items()})
print('\nverdict inputs (facts only):', json.dumps(report['verdict_inputs'], indent=1))

agreement with GT (IoU 0.5, score >= 0.5, mean over frames): {
 "raw_bgr": {
  "precision": 0.003,
  "recall": 0.041
 },
 "imagenet_rgb": {
  "precision": 0.864,
  "recall": 0.986
 }
}
raw_bgr
  dancetrack0004_00000300.jpg: GT=3 {'0.3': (47, 0.0, 0.0), '0.5': (33, 0.0, 0.0), '0.7': (20, 0.0, 0.0)}
  dancetrack0004_00000601.jpg: GT=2 {'0.3': (33, 0.0, 0.0), '0.5': (25, 0.0, 0.0), '0.7': (17, 0.0, 0.0)}
  dancetrack0004_00000902.jpg: GT=4 {'0.3': (35, 0.0, 0.0), '0.5': (26, 0.0, 0.0), '0.7': (16, 0.0, 0.0)}
  dancetrack0026_00000075.jpg: GT=14 {'0.3': (259, 0.012, 0.214), '0.5': (218, 0.014, 0.214), '0.7': (156, 0.0, 0.0)}
  dancetrack0026_00000151.jpg: GT=17 {'0.3': (303, 0.0, 0.0), '0.5': (243, 0.0, 0.0), '0.7': (182, 0.0, 0.0)}
  dancetrack0026_00000226.jpg: GT=13 {'0.3': (236, 0.008, 0.154), '0.5': (199, 0.01, 0.154), '0.7': (144, 0.007, 0.077)}
  dancetrack0063_00000250.jpg: GT=8 {'0.3': (305, 0.0, 0.0), '0.5': (236, 0.0, 0.0), '0.7': (181, 0.0, 0.0)}
  dancetrack0063_00000500.jpg: 

## Đọc kết quả

- `loaded_strict=false` → thường là sai kiến trúc giả định (depth/width/num_classes) hoặc thiếu phụ thuộc; xem `error`, **không** suy ra checkpoint hỏng.
- Đạt về phát hiện = **precision và recall cao so với GT** (IoU 0.5) trên 9 khung, không phải "có box". Lần chạy đầu chỉ đếm box: 210 box trên khung có 7 GT, nên không chứng minh được gì. Hai kiểu tiền xử lý được chạy song song; `best_variant` là kiểu khớp GT hơn, và nếu cả hai đều kém thì checkpoint chưa dùng được theo cách nạp này.
- Cả 3 ngưỡng (0.3/0.5/0.7) được in: precision thấp ở 0.3 nhưng cao ở 0.5–0.7 là bình thường với detector cho tracker.
- `training_data_documented=false` (không có `exp`/`args` trong checkpoint) = chưa biết nó đã thấy val hay chưa → muốn dùng phải tra repo DanceTrack/ByteTrack exp file trước khi gọi val là "unseen".
- Gửi lại `yolox-checkpoint-smoke.json` (trong thư mục kết quả) để đóng câu hỏi nguồn (b).